<a href="https://colab.research.google.com/github/Praim-maker/---RAG-Retrieval-Augmented-Generation-/blob/main/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22%D0%AF%D0%BA%D1%83%D0%B1%D0%BE%D0%B2_%D0%A0%D0%B0%D0%BC%D0%B8%D1%81_31_%D0%98%D0%A1%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langchain-community langchain-huggingface faiss-cpu pypdf huggingface_hub
!pip install --no-cache-dir llama-cpp-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.1/70.1 MB 241.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 289.8 MB/s eta 0:00:

In [ ]:

# ==============================================================================
import os   # Системный модуль для проверки путей и существования файлов
import sys  # Системный модуль для работы с параметрами интерпретатора
# ==============================================================================
# 2. ИМПОРТ МОДУЛЕЙ СИСТЕМЫ
# ==============================================================================
print("--- [1/4] Подключение модулей и проверка файлов... ---")
import shutil
from huggingface_hub import hf_hub_download
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import LlamaCpp

# Настройки путей
PDF_FILE = "Курсовой проект.pdf"
MODEL_FILE = "llama3.gguf"
REPO_ID = "QuantFactory/Meta-Llama-3-8B-Instruct-GGUF"
FILENAME = "Meta-Llama-3-8B-Instruct.Q4_K_M.gguf"

# Проверка наличия вашего документа на панели слева
if not os.path.exists(PDF_FILE):
    raise FileNotFoundError(f"\n❌ ОШИБКА: Файл '{PDF_FILE}' не найден на панели слева (📁)! Загрузите его заново.")

# ==============================================================================
# 3. СКАЧИВАНИЕ МОДЕЛИ LLAMA-3 (ЕСЛИ НЕ СКАЧАНА)
# ==============================================================================
if not os.path.exists(MODEL_FILE):
    print("--- 📥 Скачивание модели Llama-3 (~4.8 ГБ, ожидайте окончания)... ---")
    downloaded_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)
    shutil.copy(downloaded_path, MODEL_FILE)
    print("--- ✅ Модель успешно загружена! ---")
else:
    print("--- ✅ Модель уже загружена ранее. ---")

# ==============================================================================
# 4. ПОДГОТОВКА RAG (БАЗА ЗНАНИЙ FAISS)
# ==============================================================================
print("--- [2/4] Чтение PDF и создание базы знаний... ---")
loader = PyPDFLoader(PDF_FILE)
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
splits = text_splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
vector_store = FAISS.from_documents(splits, embeddings)

# === 5. ЗАГРУЗКА НЕЙРОСЕТИ В ПАМЯТЬ ===
print("--- [3/4] Загрузка локальной LLM... ---")
llm = LlamaCpp(
    model_path=MODEL_FILE,
    n_ctx=3000,          # Расширяем контекст
    temperature=0.1,     # Точность фактов
    max_tokens=512,      # Ограничение на длину ответа
    verbose=False
)

--- [1/4] Подключение модулей и проверка файлов... ---
--- 📥 Скачивание модели Llama-3 (~4.8 ГБ, ожидайте окончания)... ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Meta-Llama-3-8B-Instruct.Q4_K_M.gguf:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

--- ✅ Модель успешно загружена! ---
--- [2/4] Чтение PDF и создание базы знаний... ---


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

--- [3/4] Загрузка локальной LLM... ---


llama_context: n_ctx_seq (3072) < n_ctx_train (8192) -- the full capacity of the model will not be utilized


In [ ]:
# ==============================================================================
# 6. ИНТЕРАКТИВНЫЙ ИНТЕРФЕЙС ВОПРОСОВ И ОТВЕТОВ ПРЯМО В COLAB
# ==============================================================================
print("Введите ваш вопрос ниже. Для выхода введите 'выход'.")

while True:
    user_query = input("\nВаш вопрос: ")
    if user_query.lower() in ['выход', 'exit', 'quit']:
        print("Работа завершена. До свидания!")
        break
    if not user_query.strip():
        continue

    print("Бот анализирует текст курсового проекта...")

    # Шаг А: Ищем в FAISS топ-3 самых релевантных куска из PDF
    search_results = vector_store.similarity_search(user_query, k=3)
    context_text = "\n\n".join([doc.page_content for doc in search_results])

    # Шаг Б: Формируем чистый, понятный для Llama-3 промпт напрямую
    clean_prompt = f"""System: Ты — точный и полезный ИИ-ассистент. Отвечай на вопрос пользователя развернуто, структурировано и строго на русском языке, используя ТОЛЬКО предоставленный текст курсового проекта. Если в тексте нет ответа, напиши: 'В документе нет информации по этому вопросу'.

Контекст из документа:
{context_text}

User: {user_query}
Assistant:"""

    # Шаг В: Прямой вызов модели без LangChain-цепочек (Фикс абракадабры)
    raw_response = llm.invoke(clean_prompt)

    # Очищаем ответ от возможных технических хвостов разметки
    final_answer = raw_response.strip().split("User:")[0].split("System:")[0].strip()

    print(f"\nОтвет бота:\n{final_answer}")
    print("-" * 60)


Введите ваш вопрос ниже. Для выхода введите 'выход'.

Ваш вопрос: Тема курсовай в данном файле?
Бот анализирует текст курсового проекта...

Ответ бота:
В данном файле не обнаружено темы курсового проекта. Файл содержит текст, который может быть использован для создания курсового проекта, но не является его частью. Если вам нужна помощь в создании курсового проекта, пожалуйста, задайте вопросы или предоставьте дополнительную информацию о вашем проекте. Я готов помочь вам в любом месте!
------------------------------------------------------------

Ваш вопрос: Ответсвенные по практике?
Бот анализирует текст курсового проекта...
